# Metal-insulator transitions and the Hubbard-I approximation

In this tutorial, we extend our study of La$_{2}$CuO$_{4}$ by exploring how electron correlations drive a metal-insulator transition within the DMFT framework. Starting from the effective low-energy model built in the previous tutorial, we will increase the strength of the Hubbard interaction $U$, rerun the DMFT calculation, and analyze how the spectral function eveolves from metallic to insulating behavior.

Additionally, we will explore the effects of swapping to a different (cheaper) approximate impurity solver (Hubbard-I approximation). We will discuss then these approximate impurity solvers and build intutition about when they are appropriate.

Specifically, you will learn how to:

- Analyze how the local spectral function change across the metal-insulator transition.
- Interchange impurity solvers in your DMFT script.
- Study the impurity problem in the interacting limit.

In [ ]:
import numpy as np 
from common import *
import bzsummation as modest
from triqs.gf import MeshImFreq, MeshDLRImFreq
from triqs.operators import n
from triqs.plot.mpl_interface import oplot

hdf5_filename = 'data/mlwf/lco_wannier.h5'
target_density, obe = modest.one_body_elements_from_dft_converter(hdf5_filename)
E = modest.make_embedding_with_equivalences(obe.C_space)

beta = 10.0 # inverse temperature
mesh = MeshImFreq(beta, S='Fermion', n_iw=251) # Matsubara mesh

U = 5.0
h_int = U*n('up_0',0)*n('down_0',0)
solver_params = dict(n_iw=251, n_tau=2510, length_cycle=60, n_cycles = int(2e+6), n_warmup_cycles = int(1e+4), 
                     perform_tail_fit=True, fit_min_w=6, fit_max_w=10, 
                     imag_threshold = 1e-6)
# h5 read results from file

# update Sigma!
Sigma_hartree_C  = E.embed([list(solver_results.Sigma_Hartree)])
Sigma_dynamic_C  = E.embed([solver_results.Sigma_dynamic]) 

# update mu!
mu     = modest.find_chemical_potential(target_density, obe, Sigma_dynamic_C, Sigma_hartree_C, verbosity=False)

# update Gloc!
Gloc   = E.extract(modest.gloc(obe, mu, Sigma_dynamic_C, Sigma_hartree_C))[0]

# update hloc0! εd - μ
hloc0 = E.extract(modest.impurity_levels(obe) - mu)[0]

# update Δ!
Delta_iw = modest.extract_delta(hloc0, Gloc, solver_results.Sigma_dynamic, list(solver_results.Sigma_Hartree))


n_dmft_loops = 10
for n_iter in range(n_dmft_loops):
    print(f"DMFT iteration= {n_iter}")

    # solve!
    solver_results = solve(Delta_iw, hloc0, h_int, **solver_params)

    # update Sigma!
    Sigma_hartree_C  = E.embed([list(solver_results.Sigma_Hartree)])
    Sigma_dynamic_C  = E.embed([solver_results.Sigma_dynamic]) 

    # update mu!
    mu     = modest.find_chemical_potential(target_density, obe, Sigma_dynamic_C, Sigma_hartree_C, verbosity=False)

    # update Gloc!
    Gloc   = E.extract(modest.gloc(obe, mu, Sigma_dynamic_C, Sigma_hartree_C))[0]

    # update hloc0! εd - μ
    hloc0 = E.extract(modest.impurity_levels(obe) - mu)[0]

    # update Δ!
    Delta_iw = modest.extract_delta(hloc0, Gloc, solver_results.Sigma_dynamic, list(solver_results.Sigma_Hartree))

    print(f"Δn = |n_lattice - n_impurity| = {abs(Gloc.total_density()-solver_results.G_iw.total_density())}")